In [ ]:
# Import libraries and dataset
# Implementation comes this cell
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense

# import data
df_penguins = pd.read_csv('penguins_size.csv')

# Data examination (You can do as you want)
# Implementation comes this cell
# df_penguins.info()
#print(df_penguins.head())
#print(df_penguins.isnull().sum())
#df_penguins = df_penguins.dropna()
# print(df_penguins.isnull().sum())

# for col in ['species', 'island', 'sex']:
#     print(df_penguins[col].value_counts(dropna=False))

for col in ['species', 'island', 'sex']:
    print(col, df_penguins[col].unique())

# for col in df_penguins.select_dtypes(include='object').columns: # include=['object', 'string']
#     print(col, df_penguins[col].unique())

# Treat the invalid '.' in sex as missing, then drop rows with any null
df_penguins['sex'] = df_penguins['sex'].replace('.', np.nan)

# df_penguins = df_penguins.dropna().reset_index(drop=True)
df_penguins = df_penguins.dropna().copy()

import seaborn as sns
import matplotlib.pyplot as plt

sns.pairplot(df_penguins.dropna(), hue='species', diag_kind='kde')
plt.show()

df_penguins['species'].value_counts().plot(kind='bar', title='Penguins per species')
plt.ylabel('Count')
plt.show()

# Categorical varibles: Encode columns Species, Island and Sex
# species = Adélie, Chinstrap and Gentoo
# sex = MALE, FEMALA
# island = Torgersen, Biscoe and Dream

# Implementation comes this cell

# Group numeric and categorical features separately for preprocessing
categorical_cols    = ["island", "sex"]
numeric_cols    = ["culmen_length_mm", "culmen_depth_mm", "flipper_length_mm", "body_mass_g"]

# Convert categorical features to dummy variables
# dummies_categorical_df = pd.get_dummies(df_penguins[categorical_cols], dtype=int, drop_first=True)
dummies_categorical_df = pd.get_dummies(df_penguins[categorical_cols], dtype=int)
# X = pd.concat([df_penguins[numeric_cols], dummies_categorical_df], axis=1)

# Scale all the numeric features using StandardScaler
scaler_x = StandardScaler()
scaled_numeric = scaler_x.fit_transform(df_penguins[numeric_cols]) # returns NumPy array

# Prepare features (X)
X_scaled = np.concatenate((scaled_numeric, dummies_categorical_df.to_numpy()), axis=1)

# Prepare target variable (y)
y = df_penguins["species"].to_numpy()
enc = LabelEncoder()
y_labeled = enc.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_labeled, test_size=0.2, random_state=42, stratify=y_labeled
)

# # Scale only the 4 numeric columns; fit on training data only
# scaler_x = StandardScaler()
# X_train[:, :4] = scaler_x.fit_transform(X_train[:, :4])
# X_test[:, :4] = scaler_x.transform(X_test[:, :4])


# # Test version: encode first, scale after splitting
# # Convert categorical features to dummy variables
# dummies_categorical_df_v2 = pd.get_dummies(df_penguins[categorical_cols], dtype=int)

# # Prepare features (X): unscaled numeric columns first, then dummies
# X_unscaled_v2 = np.concatenate((df_penguins[numeric_cols].to_numpy(),
#                                 dummies_categorical_df_v2.to_numpy()), axis=1)

# # Prepare target variable (y)
# enc_v2 = LabelEncoder()
# y_labeled_v2 = enc_v2.fit_transform(df_penguins["species"])

# # Split first
# X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
#     X_unscaled_v2, y_labeled_v2, test_size=0.2, random_state=42, stratify=y_labeled_v2
# )

# # Scale only the first 4 columns (numeric features); fit on training data only
# scaler_v2 = StandardScaler()
# X_train_v2[:, :4] = scaler_v2.fit_transform(X_train_v2[:, :4])
# X_test_v2[:, :4] = scaler_v2.transform(X_test_v2[:, :4])

# print("Train:", X_train_v2.shape, "Test:", X_test_v2.shape)   # (266, 9) and (67, 9)
# print("NaNs in X_train_v2:", np.isnan(X_train_v2).sum())       # 0



n_features = X_train.shape[1]   # 9
n_classes = len(enc.classes_)   # 3

layer_nodes_combo_options = [[2], [3], [4], [8], [4, 2]]
epoch_options = [10, 50, 100]
n_runs = 5   # train each config 5 times with different seeds

results_table = []

for layer_nodes_combo in layer_nodes_combo_options:
    for n_epochs in epoch_options:
        test_acc_runs = []   # this config's test accuracy across all runs

        for run in range(n_runs):
            tf.keras.utils.set_random_seed(run)   # seeds 0, 1, 2, 3, 4

            inputs = Input(shape=(n_features,))
            x = inputs
            for n_nodes in layer_nodes_combo:
                x = Dense(n_nodes, activation='relu')(x)
            outputs = Dense(n_classes, activation='softmax')(x)
            model = Model(inputs=inputs, outputs=outputs)
            model.compile(optimizer='adam',
                          loss='sparse_categorical_crossentropy',
                          metrics=['accuracy'])

            model.fit(X_train, y_train, epochs=n_epochs, verbose=0)
            test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
            test_acc_runs.append(test_acc)

        avg_acc = np.mean(test_acc_runs)
        std_acc = np.std(test_acc_runs)

        results_table.append((layer_nodes_combo, n_epochs, avg_acc, std_acc))
        print(f"Layer/Nodes: {layer_nodes_combo}, Epochs: {n_epochs} → "
              f"Avg Test Accuracy: {avg_acc:.4f} (± {std_acc:.4f} across {n_runs} runs)")

# results_df = pd.DataFrame(results_table,
#                           columns=["layer_nodes", "epochs", "avg_test_acc", "std_test_acc"])
# results_df.sort_values("avg_test_acc", ascending=False)



species ['Adelie' 'Chinstrap' 'Gentoo']
island ['Torgersen' 'Biscoe' 'Dream']
sex ['MALE' 'FEMALE' nan '.']
Layer/Nodes: [2], Epochs: 10 → Avg Test Accuracy: 0.4418 (± 0.2344 across 5 runs)
Layer/Nodes: [2], Epochs: 50 → Avg Test Accuracy: 0.8179 (± 0.0466 across 5 runs)
Layer/Nodes: [2], Epochs: 100 → Avg Test Accuracy: 0.9701 (± 0.0163 across 5 runs)
Layer/Nodes: [3], Epochs: 10 → Avg Test Accuracy: 0.5731 (± 0.1750 across 5 runs)
Layer/Nodes: [3], Epochs: 50 → Avg Test Accuracy: 0.9672 (± 0.0358 across 5 runs)
Layer/Nodes: [3], Epochs: 100 → Avg Test Accuracy: 0.9940 (± 0.0073 across 5 runs)
Layer/Nodes: [4], Epochs: 10 → Avg Test Accuracy: 0.6896 (± 0.1158 across 5 runs)
Layer/Nodes: [4], Epochs: 50 → Avg Test Accuracy: 0.9642 (± 0.0202 across 5 runs)
Layer/Nodes: [4], Epochs: 100 → Avg Test Accuracy: 0.9970 (± 0.0060 across 5 runs)
Layer/Nodes: [8], Epochs: 10 → Avg Test Accuracy: 0.7015 (± 0.1280 across 5 runs)
Layer/Nodes: [8], Epochs: 50 → Avg Test Accuracy: 0.9881 (± 0.0174 ac